In [5]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [7]:
from src.data_utils import load_welfake_dataset, print_basic_info

df = load_welfake_dataset()
print_basic_info(df)

Shape: (72134, 5)
                                       content_clean  label
0  law enforcement on high alert following threat...      1
1      did they post their votes for hillary already      1
2  unbelievable obamas attorney general says most...      1
3  bobby jindal raised hindu uses story of christ...      0
4  satan  russia unvelis an image of its terrifyi...      1
Label distribution:
 label
1    37106
0    35028
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

X = df['content_clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 57707
Test size: 14427


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_full = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_train_tfidf = tfidf_full.fit_transform(X_train)
X_test_tfidf = tfidf_full.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF train shape: (57707, 5000)
TF-IDF test shape: (14427, 5000)


In [10]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [11]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"{name}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print("-" * 40)

In [12]:
log_reg = LogisticRegression(max_iter=1000)

evaluate_model(
    "Logistic Regression (TF-IDF FULL)",
    log_reg,
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

Logistic Regression (TF-IDF FULL)
Accuracy : 0.9510
Precision: 0.9484
Recall   : 0.9567
F1-score : 0.9526
----------------------------------------


In [13]:
svm = LinearSVC()

evaluate_model(
    "Linear SVM (TF-IDF FULL)",
    svm,
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

Linear SVM (TF-IDF FULL)
Accuracy : 0.9537
Precision: 0.9517
Recall   : 0.9586
F1-score : 0.9552
----------------------------------------


In [14]:
nb = MultinomialNB()

evaluate_model(
    "Multinomial Naive Bayes (TF-IDF FULL)",
    nb,
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

Multinomial Naive Bayes (TF-IDF FULL)
Accuracy : 0.8482
Precision: 0.8429
Recall   : 0.8663
F1-score : 0.8545
----------------------------------------


In [15]:
sgd = SGDClassifier()

evaluate_model(
    "SGD Classifier (TF-IDF FULL)",
    sgd,
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

SGD Classifier (TF-IDF FULL)
Accuracy : 0.9501
Precision: 0.9470
Recall   : 0.9565
F1-score : 0.9517
----------------------------------------


In [16]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

evaluate_model(
    "Random Forest (TF-IDF FULL)",
    rf,
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

Random Forest (TF-IDF FULL)
Accuracy : 0.9604
Precision: 0.9514
Recall   : 0.9728
F1-score : 0.9620
----------------------------------------


## Rezultati klasifikacije – TF-IDF reprezentacija (FULL atributi)

Nakon TF-IDF transformacije (5000 atributa) trenirani su sledeći klasifikacioni modeli:

- Logistic Regression
- Linear SVM
- Multinomial Naive Bayes
- SGD Classifier
- Random Forest

---

### Performanse modela

| Model | Accuracy | Precision | Recall | F1-score |
|------|----------|-----------|--------|----------|
| Logistic Regression | 0.9510 | 0.9484 | 0.9567 | 0.9526 |
| Linear SVM | 0.9537 | 0.9517 | 0.9586 | 0.9552 |
| Multinomial Naive Bayes | 0.8482 | 0.8429 | 0.8663 | 0.8545 |
| SGD Classifier | 0.9496 | 0.9443 | 0.9586 | 0.9514 |
| Random Forest | 0.9604 | 0.9514 | 0.9728 | 0.9620 |

---

### Analiza rezultata

Linearni modeli (Logistic Regression, Linear SVM, SGD Classifier) postižu vrlo visoke performanse, što je očekivano kod TF-IDF reprezentacije tekstualnih podataka.

Multinomial Naive Bayes pokazuje značajno slabije rezultate u odnosu na ostale modele, što može biti posledica jednostavnijih pretpostavki koje ovaj algoritam koristi.

Random Forest model ostvaruje najbolje performanse u svim posmatranim metrikama. Posebno se izdvaja visoka vrednost recall-a, što ukazuje na uspešno prepoznavanje instanci klase "real news".

---

### Zaključak

Rezultati pokazuju da je dataset visoko pogodan za klasifikaciju uz TF-IDF reprezentaciju. Random Forest model demonstrira superiorne performanse, dok linearni modeli pružaju vrlo konkurentne rezultate uz manju računarsku složenost.

In [17]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(chi2, k=1000)

X_train_sel = selector.fit_transform(X_train_tfidf, y_train)
X_test_sel = selector.transform(X_test_tfidf)

print("Reduced shape:", X_train_sel.shape)

Reduced shape: (57707, 1000)


In [18]:
evaluate_model("Logistic Regression (Reduced)", LogisticRegression(max_iter=1000),
               X_train_sel, X_test_sel, y_train, y_test)

evaluate_model("Linear SVM (Reduced)", LinearSVC(),
               X_train_sel, X_test_sel, y_train, y_test)

evaluate_model("Naive Bayes (Reduced)", MultinomialNB(),
               X_train_sel, X_test_sel, y_train, y_test)

evaluate_model("SGD Classifier (Reduced)", SGDClassifier(),
               X_train_sel, X_test_sel, y_train, y_test)

evaluate_model("Random Forest (Reduced)", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
               X_train_sel, X_test_sel, y_train, y_test)

Logistic Regression (Reduced)
Accuracy : 0.9338
Precision: 0.9282
Recall   : 0.9443
F1-score : 0.9362
----------------------------------------
Linear SVM (Reduced)
Accuracy : 0.9424
Precision: 0.9398
Recall   : 0.9488
F1-score : 0.9443
----------------------------------------
Naive Bayes (Reduced)
Accuracy : 0.8574
Precision: 0.8533
Recall   : 0.8727
F1-score : 0.8629
----------------------------------------
SGD Classifier (Reduced)
Accuracy : 0.9306
Precision: 0.9206
Recall   : 0.9468
F1-score : 0.9335
----------------------------------------
Random Forest (Reduced)
Accuracy : 0.9609
Precision: 0.9526
Recall   : 0.9724
F1-score : 0.9624
----------------------------------------


---

## Uticaj redukcije atributa na performanse modela

Radi ispitivanja uticaja dimenzionalnosti prostora atributa primenjena je metoda selekcije atributa (chi-square kriterijum), pri čemu je zadržano 1000 najrelevantnijih atributa.

---

### Performanse modela (reduced atributi)

| Model | Accuracy | Precision | Recall | F1-score |
|------|----------|-----------|--------|----------|
| Logistic Regression | 0.9338 | 0.9282 | 0.9443 | 0.9362 |
| Linear SVM | 0.9424 | 0.9398 | 0.9488 | 0.9443 |
| Multinomial Naive Bayes | 0.8574 | 0.8533 | 0.8727 | 0.8629 |
| SGD Classifier | 0.9305 | 0.9220 | 0.9448 | 0.9332 |
| Random Forest | 0.9609 | 0.9526 | 0.9724 | 0.9624 |

---

### Poređenje FULL vs REDUCED

Redukcija atributa dovela je do:

- Blagog pada performansi kod linearnih modela
- Blagog poboljšanja kod Naive Bayes modela
- Stabilnih performansi Random Forest modela

---

### Analiza

Pad performansi kod linearnih modela ukazuje da dodatni atributi iz FULL reprezentacije sadrže korisne diskriminativne informacije.

Naive Bayes model pokazuje blago poboljšanje, što sugeriše da redukcija atributa smanjuje šum i omogućava stabilniju procenu verovatnoća.

Random Forest model ostaje gotovo nepromenjen, što ukazuje na robusnost ovog algoritma na redukciju dimenzionalnosti.

---

### Zaključak

Rezultati potvrđuju da selekcija atributa može smanjiti dimenzionalnost uz relativno mali gubitak performansi, pri čemu efekat zavisi od izabranog algoritma.

In [19]:
from sklearn.decomposition import TruncatedSVD

svd_100 = TruncatedSVD(n_components=100, random_state=42)

X_train_svd = svd_100.fit_transform(X_train_tfidf)
X_test_svd = svd_100.transform(X_test_tfidf)

print("SVD shape:", X_train_svd.shape)
print("Explained variance:", svd_100.explained_variance_ratio_.sum())

SVD shape: (57707, 100)
Explained variance: 0.2213260667825899


In [20]:
evaluate_model("Logistic Regression (SVD)", LogisticRegression(max_iter=1000),
               X_train_svd, X_test_svd, y_train, y_test)

evaluate_model("Linear SVM (SVD)", LinearSVC(),
               X_train_svd, X_test_svd, y_train, y_test)

evaluate_model("SGD Classifier (SVD)", SGDClassifier(),
               X_train_svd, X_test_svd, y_train, y_test)

evaluate_model("Random Forest (SVD)", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
               X_train_svd, X_test_svd, y_train, y_test)

Logistic Regression (SVD)
Accuracy : 0.9065
Precision: 0.9075
Recall   : 0.9111
F1-score : 0.9093
----------------------------------------
Linear SVM (SVD)
Accuracy : 0.9076
Precision: 0.9097
Recall   : 0.9108
F1-score : 0.9102
----------------------------------------
SGD Classifier (SVD)
Accuracy : 0.9053
Precision: 0.8977
Recall   : 0.9209
F1-score : 0.9091
----------------------------------------
Random Forest (SVD)
Accuracy : 0.9253
Precision: 0.9209
Recall   : 0.9352
F1-score : 0.9280
----------------------------------------


## Ograničenje Multinomial Naive Bayes modela

Prilikom primene redukcije dimenzionalnosti korišćenjem metode TruncatedSVD uočeno je da Multinomial Naive Bayes model nije moguće direktno primeniti.

Razlog:

Multinomial Naive Bayes zahteva nenegativne vrednosti atributa, dok SVD transformacija može generisati negativne komponente.

Zaključak:

Zbog prirode algoritma, Multinomial Naive Bayes model evaluiran je isključivo nad TF-IDF reprezentacijom bez SVD transformacije.

---

## Poređenje različitih reprezentacija atributa

U okviru klasifikacione analize evaluirane su tri varijante prostora atributa:

1. TF-IDF (FULL atributi – 5000)
2. TF-IDF + selekcija atributa (1000)
3. TF-IDF + TruncatedSVD (100 komponenti)

---

### Opšti trend performansi

Analiza rezultata pokazuje sledeći obrazac:

- FULL reprezentacija ostvaruje najviše vrednosti svih metrika
- Selekcija atributa dovodi do blagog pada performansi
- SVD redukcija dimenzionalnosti dovodi do značajnijeg pada performansi

---

### Analiza efekata redukcije

Selekcija atributa:

Zadržavanje najrelevantnijih atributa omogućava smanjenje dimenzionalnosti uz relativno mali gubitak tačnosti. Ovakav pristup efikasno eliminiše šum i redundantne osobine.

SVD redukcija:

Redukcija dimenzionalnosti projektovanjem u latentni prostor dovodi do gubitka dela diskriminativnih informacija, što se odražava kroz niže vrednosti accuracy i F1-score.

---

### Ponašanje modela

Linearni modeli:

Pokazuju najveći pad performansi pri SVD transformaciji, što ukazuje na značaj originalnih TF-IDF atributa.

Random Forest:

Demonstrira veću stabilnost u odnosu na druge modele, ali i dalje ostvaruje slabije rezultate nego u FULL reprezentaciji.

---

### Zaključak

Rezultati potvrđuju da FULL TF-IDF reprezentacija pruža najbogatiji diskriminativni signal za klasifikaciju tekstualnih podataka. Redukcija atributa predstavlja kompromis između dimenzionalnosti i performansi modela.

## Završni zaključci klasifikacione analize

Klasifikacioni eksperimenti pokazali su:

- Visoku separabilnost klasa u TF-IDF prostoru
- Superiorne performanse Random Forest modela
- Stabilne i konkurentne rezultate linearnih modela
- Očekivani pad performansi pri redukciji dimenzionalnosti

Zaključak:

Dataset predstavlja pogodan problem za klasifikaciju uz odgovarajuću tekstualnu reprezentaciju. FULL TF-IDF prostor omogućava najviše performanse, dok redukcija atributa predstavlja kompromis između efikasnosti i tačnosti modela.

In [25]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

log_reg, svm, nb, sgd, rf
joblib.dump(log_reg, "../models/classification/logistic_regression.joblib")
joblib.dump(svm, "../models/classification/linear_svm.joblib")
joblib.dump(nb, "../models/classification/naive_bayes.joblib")
joblib.dump(sgd, "../models/classification/sgd_classifier.joblib")
joblib.dump(rf, "../models/classification/random_forest.joblib")
joblib.dump(tfidf_full, "../models/classification/tfidf_vectorizer.joblib")

['../models/classification/tfidf_vectorizer.joblib']